# Programmatic Workflow & Reproducing Runs

This notebook shows how to use **diag_evo** without the interactive widget UI.
Two workflows are demonstrated:

1. **From a saved `run_config.json`** — reproduce a previous run
2. **From scratch** — build the configuration entirely in code

Both approaches produce exactly the same results as the interactive notebook.

---

## Workflow A — Reproduce from a saved `run_config.json`

Pressing the **Forecast Setup** button in
[`forecast_evolution.ipynb`](forecast_evolution.ipynb) writes a
`run_config.json` to `<base_path>/run_configs/<param>_<date>_<area>/`.
That file captures all settings and can be loaded later to reproduce
the exact same run.

### What the JSON contains

```json
{
  "param": "2t",
  "levtype": "sfc",
  "level": null,
  "valid_date": "2026-04-07T12:00:00",
  "area_sub": [50.905, -1.143, 44.905, 4.857],
  "point": [47.905, 1.857],
  "max_days": 3,
  "step_interval": 24,
  "forecast_dates": ["2026-04-06T12:00:00", "2026-04-05T12:00:00"],
  "forecast_steps": [24, 48, 72],
  "selected_models": ["IFS Control", "AIFS Single", "AIFS ENS"],
  "n_members": 50,
  "area_avg_mode": "integrate",
  "acc_period": 24
}
```

`acc_period` is set to `null` automatically for non-accumulated
parameters, and is ignored unless the variable is accumulated.

In [ ]:
import os, sys, glob

notebook_dir = "/path/to/diag_evo"  # <-- Update to the directory containing the diag_evo package
sys.path.insert(0, notebook_dir)

base_path = "/path/to/your/data_files"  # <-- Update to where retrieved GRIB data should be cached

from diag_evo import (
    load_run_config,
    build_widgets_dict,
    register_custom_model,
    retrieve_and_store_data,
    setup_data_directories,
    get_area_string,
    plot_forecast_evolution,
    plot_forecast_evolution_static,
    plot_field_map,
    plot_obs_map,
    plot_analysis_map,
)

### A1 — Find and load a saved config

In [ ]:
# Point to a specific config file:
config_path = "<base_path>/run_configs/<param>_<date>_<area>/run_config.json  # <-- update to actual path"

# Or discover all saved configs:
configs = sorted(glob.glob(os.path.join(base_path, "run_configs/*/run_config.json")))
for c in configs:
    print(c)
config_path = configs[-1]

print(f"Loading: {config_path}")

widgets_dict = load_run_config(config_path)
config = widgets_dict['config']

print(f"Parameter : {config['param']}")
print(f"Valid date: {config['valid_date']}")
print(f"Area      : {config['area_sub']}")
print(f"Point     : {config.get('point')}")
print(f"Max days  : {config['max_days']}")
print(f"Step int. : {config['step_interval']}")
print(f"Models    : {config['selected_models']}")

### A2 — Retrieve data (uses cache)

In [ ]:
plot_data = retrieve_and_store_data(widgets_dict, base_path)

# Build the plot directory path
area_str = get_area_string(config['area_sub'])
date_str = config['valid_date'].strftime("%Y%m%d")
time_str = f"{config['valid_date'].hour:02d}00"
_, _, plot_dir = setup_data_directories(
    base_path, config['param'], area_str, date_str, time_str
)

### A3 — Plot

In [ ]:
# Interactive (Plotly)
plot_forecast_evolution(plot_data, widgets_dict, plot_dir, export_html=True)

# Static (Matplotlib)
plot_forecast_evolution_static(plot_data, widgets_dict, plot_dir, export_png=True)

In [ ]:
plot_forecast_evolution_static(plot_data, widgets_dict, plot_dir, pctl_method="linear", export_png=True)

In [ ]:
plot_field_map(plot_data, widgets_dict, 'AIFS ENS', member=4, step=12,
                           plot_mode='field', show_gridpoints=True, overlay_obs=True,
                           plot_radius=-0.25, export_png=True)

In [ ]:
# Map plots
for step in [24]:
    for model in config['selected_models']:
        try:
            plot_field_map(plot_data, widgets_dict, model, step=step,
                           plot_mode='field', overlay_obs=True,
                           plot_radius=2, export_png=True)
        except Exception as e:
            print(f"  skip {model} T+{step}h: {e}")
    
try:
    plot_obs_map(plot_data, widgets_dict, plot_radius=2, export_png=True)
except Exception as e:
    print(f"  skip obs map: {e}")

try:
    plot_analysis_map(plot_data, widgets_dict,
                      overlay_obs=True, plot_radius=3, export_png=True)
except Exception as e:
    print(f"  skip analysis map: {e}")

In [ ]:
from diag_evo import add_nb_ens_to_plot_data

nb_path = "/ec/vol/destine/neighbourhood/percentile"

plot_data_nb, widgets_dict = add_nb_ens_to_plot_data(plot_data, widgets_dict, nb_path)

# Interactive Plotly plot — NB ENS shown alongside selected models
plot_forecast_evolution(plot_data_nb, widgets_dict, plot_dir, export_html=True)


In [ ]:
plot_forecast_evolution_static(plot_data_nb, widgets_dict, plot_dir, export_png=True)

---

## Workflow B — Build configuration from scratch (no UI)

You can bypass the widget interface entirely and build the configuration
in code.  This is useful for:

- **Batch processing** — loop over multiple parameters, dates or regions
- **Scripted pipelines** — integrate into a larger workflow
- **HPC jobs** — run via `sbatch` without a Jupyter session

You can select either an **area** or a single **point** (area is
auto-derived as a ±3° bounding box when only a point is given), and
you can register **custom MARS models** with
`register_custom_model()`. The config dict is wrapped into a
`widgets_dict` via `build_widgets_dict()`, which is exactly what the
"Forecast Setup" button does in the interactive notebook.

### B1 — (Optional) Register a custom MARS model

Skip this cell if you only want to use the predefined models.

In [ ]:
# Example: register a custom MARS model (class=rd) under a friendly name.
# The name you choose here must then appear in SELECTED_MODELS below.
register_custom_model(
    name="AIFS_Single_version2",
    retrieval_args={
        "class": "ai",
        "type": "fc",
        "stream": "oper",
        "expver": "0105",
        "model":"aifs-single"
    },
    is_ensemble=False,
    color="#ff6600",
)

### B2 — Define the configuration and build `widgets_dict`

Choose **either** `point` (area auto-derived as ±3° box) **or** a full
`area_sub` bounding box. `forecast_dates` / `forecast_steps` are
auto-derived from `valid_date`, `max_days` and `step_interval`.
`acc_period` is only used when the parameter is accumulated (e.g. `tp`).

In [ ]:
from datetime import datetime

# ---- Option 1: point-based (area_sub auto-derived to a ±3° box) ----
config = {
    "param":           "2t",
    "levtype":         "sfc",
    "level":           None,
    "valid_date":      datetime(2026, 4, 16, 12),
    "point":           [51.0, -1.5],          # ~Salisbury, UK
    "max_days":        4,
    "step_interval":   12,
    "selected_models": [
        "IFS Control",
        "AIFS Single",
        "IFS ENS",
        "AIFS ENS",
        "AIFS_Single_version2",                  # <-- uncomment if B1 was run
    ],
    "n_members":       50,
    "area_avg_mode":   "station_nearest",
    # "acc_period":    6,                     # only used for accumulated params (tp)
}

# ---- Option 2: full area (uncomment and remove "point" above) ----
# config = { ..., "area_sub": [52.0, -5.0, 50.0, 2.0], ... }

# Mimic pressing "Forecast Setup" in the UI:
# normalises longitudes, derives missing area_sub / forecast_dates,
# and wraps everything in a widgets_dict-compatible structure.
widgets_dict = build_widgets_dict(config)
print(config)

### B3 — Retrieve data

In [ ]:
plot_data = retrieve_and_store_data(widgets_dict, base_path)

config = widgets_dict["config"]
area_str = get_area_string(config["area_sub"])
date_str = config["valid_date"].strftime("%Y%m%d")
time_str = f"{config['valid_date'].hour:02d}00"
_, _, plot_dir = setup_data_directories(
    base_path, config["param"], area_str, date_str, time_str
)

### B4 — Plot

In [ ]:
# Forecast evolution plots
plot_forecast_evolution(plot_data, widgets_dict, plot_dir, export_html=True)
plot_forecast_evolution_static(plot_data, widgets_dict, plot_dir, export_png=True)

In [ ]:
# Map plots
for step in [24, 48]:
    for model in SELECTED_MODELS:
        try:
            plot_field_map(plot_data, widgets_dict, model, step=step,
                           plot_mode='field', overlay_obs=True,
                           plot_radius=4, export_png=True)
        except Exception as e:
            print(f"  skip {model} T+{step}h: {e}")

try:
    plot_obs_map(plot_data, widgets_dict, plot_radius=3, export_png=True)
except Exception as e:
    print(f"  skip obs map: {e}")

try:
    plot_analysis_map(plot_data, widgets_dict,
                      overlay_obs=True, plot_radius=3, export_png=True)
except Exception as e:
    print(f"  skip analysis map: {e}")

### B5 — Optional: colour overrides

Set custom colours before re-running the plot cells above.

In [ ]:
# widgets_dict['config']['model_colors'] = {
#     'IFS Control': '#e0a7e9',
#     'AIFS Single': '#93190c',
#     'IFS ENS':     '#1e837f',
#     'AIFS ENS':    '#2c32a0',
# }
# plot_forecast_evolution_static(plot_data, widgets_dict, plot_dir, export_png=True)

---

### Command-line alternative

The same workflow can be run entirely from the command line:

```bash
# From a saved config
python examples/run_from_config.py <base_path>/run_configs/<param>_<date>_<area>/run_config.json

# Or use one of the self-contained example scripts (each showcases different features)
python examples/run_area_wind_france.py        # area mode, custom model, colour overrides, pctl_method=linear
python examples/run_point_precip_lisbon.py     # tp accumulation, NB ENS, --replay flag for config replay
python examples/run_area_t850_central_europe.py  # pressure-level param, station_nearest avg, dual pctl-method
```

See `examples/run_from_config.py --help` for CLI flags (`--no-maps`, `--map-steps`, etc.).